In [1]:
# !pip install plotly numpy pandas matplotlib
# !pip install --upgrade nbformat
# !pip install pySankey seaborn

In [2]:
OriginalProjectPath = "../../2018 FreudMeOutProject"
SecondProjectPath = "../../2019 Freud2.0"
MobileProjectPath = "../../2020 Mobile"
VrProjectPath = "../../2023 Affective Game VR"
TestProjectPath = "../."

In [3]:
def getProjectNameFromPath(path):
    return path.split("/")[-1]

In [4]:
import os
import plotly.graph_objects as go
from collections import defaultdict

In [5]:
def human_readable_size(size_bytes):
    """
    Convert a file size in bytes to a human-readable string format.
    e.g., 2048 -> '2.0 KB'
    """
    if size_bytes == 0:
        return "0 B"
    units = ["B", "KB", "MB", "GB", "TB"]
    i = 0
    while size_bytes >= 1024 and i < len(units) - 1:
        size_bytes /= 1024.0
        i += 1
    return f"{size_bytes:.1f} {units[i]}"

In [6]:
class DirectoryNode:
    # Class to handle node in directory structure
    # Class should have absolute path, relative path, reference to parent node, reference to children nodes, list of files,
    # and cached size. If size is not cached, it should be calculated on demand. Size equals to sum of all files in the node and all children nodes.

    def __init__(self, absolute_path, relative_path="", parent=None):
        self.absolute_path = absolute_path
        self.relative_path = relative_path
        self.parent = parent
        self.children = []
        self.files = []
        self._cached_size = None
        self._cached_file_count = None

    def get_name(self):
        return os.path.basename(self.absolute_path)
    
    def add_file(self, file_name):
        self.files.append(file_name)

    def add_child(self, child_node):
        self.children.append(child_node)

    def get_size(self):
        if self._cached_size is not None:
            return self._cached_size
        total_size = sum(os.path.getsize(os.path.join(self.absolute_path, f)) for f in self.files)
        for child in self.children:
            total_size += child.get_size()
        self._cached_size = total_size
        return total_size
    
    #element count is number of files in the node and all children nodes
    def get_file_count(self):
        if self._cached_file_count is not None:
            return self._cached_file_count
        total_count = len(self.files)
        for child in self.children:
            total_count += child.get_file_count()
        self._cached_file_count = total_count
        return total_count
    
    #preety print all properties, by printing all properties of the class, and tabbed all properties of the children
    def pretty_print(self, indent=0):
        print(" " * indent + f"Node: {self.get_name()} (Size: {human_readable_size(self.get_size())})")
        for file in self.files:
            print(" " * (indent + 2) + f"File: {file}")
        for child in self.children:
            child.pretty_print(indent + 2)


In [7]:
def generate_directory_tree(base_path):
    """
    Generates a directory tree structure starting from the base path.
    Returns the root DirectoryNode.
    """
    absolute_path = os.path.abspath(base_path)
    # print(f"Generating directory tree for: {absolute_path}")
    root_node = DirectoryNode(absolute_path=absolute_path, relative_path=os.path.basename(absolute_path), parent=None)
    nodes = {base_path: root_node}

    for root, dirs, files in os.walk(base_path):
        current_node = nodes[root]
        for file in files:
            current_node.add_file(file)
        for dir_name in dirs:
            dir_path = os.path.join(root, dir_name)
            relative_path = os.path.relpath(dir_path, base_path)
            child_node = DirectoryNode(absolute_path=dir_path, relative_path=relative_path, parent=current_node)
            current_node.add_child(child_node)
            nodes[dir_path] = child_node

    return root_node

In [8]:
#create and enum to switch between size and file count
class DisplayMode:
    SIZE = "size"
    FILE_COUNT = "file_count"

In [9]:
def map(inmin, inmax, outmin, outmax, value):
    return (value - inmin) / (inmax - inmin) * (outmax - outmin) + outmin

In [10]:
def generate_sankey_from_tree_dfs(root, display_mode=DisplayMode.FILE_COUNT, max_depth=None, skip_threshold=0.01):
    """
    Converts a DirectoryNode tree into Plotly Sankey data using DFS traversal.

    Args:
        root (DirectoryNode): The root of the directory tree.
        display_mode (DisplayMode): Either SIZE or FILE_COUNT.
        max_depth (int, optional): Max depth to include in traversal.

    Returns:
        Tuple: (labels, source, target, value, depth_to_nodeCount)
    """
    if max_depth is not None and max_depth < 2:
        raise ValueError("max_depth must be at least 2 to include root and children nodes.")

    labels = []
    index_map = {}
    source = []
    target = []
    value = []
    depth_to_nodeCount = defaultdict(int)

    def add_label(node):
        rel = node.relative_path or "root"
        if rel not in index_map:
            appendix = human_readable_size(node.get_size()) if display_mode == DisplayMode.SIZE else str(node.get_file_count())
            labels.append(f"{node.get_name()} ({appendix})")
            index_map[rel] = len(labels) - 1
        return index_map[rel]

    def should_include_node(node, depth):
        if max_depth is not None and depth > max_depth - 1:
            return False
        size = node.get_size() if display_mode == DisplayMode.SIZE else node.get_file_count()
        parent = node.parent
        if parent:
            parent_size = parent.get_size() if display_mode == DisplayMode.SIZE else parent.get_file_count()
            if size < parent_size * skip_threshold:
                return False
        return True

    def add_node(node, depth=0):
        if not should_include_node(node, depth):
            return

        parent_idx = add_label(node)
        depth_to_nodeCount[depth] += 1

        for child in node.children:
            if not should_include_node(child, depth + 1):
                continue
            child_idx = add_label(child)
            source.append(parent_idx)
            target.append(child_idx)
            val = child.get_size() if display_mode == DisplayMode.SIZE else child.get_file_count()
            value.append(val)
            add_node(child, depth + 1)

    add_node(root)
    return labels, source, target, value


In [11]:
import numpy as np
def plot_sankey(labels, source, target, value, title="Directory Size Sankey Diagram",height=800, width=1800):
    fig = go.Figure(go.Sankey(
        arrangement="snap",
        node=dict(
            label=labels,
            align="left",
            ),
        link=dict(
            source=source,
            target=target,
            value=value,
        )
    ))
    fig.update_layout(title_text=title, font_size=14,height=height, width=width)
    fig.show()

In [12]:
plot_sankey(*generate_sankey_from_tree_dfs(generate_directory_tree(OriginalProjectPath), display_mode=DisplayMode.SIZE,max_depth=12,skip_threshold=0.001),title=getProjectNameFromPath(OriginalProjectPath) + " ( Folder sizes )")
plot_sankey(*generate_sankey_from_tree_dfs(generate_directory_tree(OriginalProjectPath), display_mode=DisplayMode.FILE_COUNT,max_depth=12,skip_threshold=0.001),title=getProjectNameFromPath(OriginalProjectPath) + " ( File count )")

## wnioski
Powyższe wykresy pokazują że głębokość 5 wnosi niewiele, a poniżej to już nic. Stąd ograniczenie głębokości do 5

In [17]:
diagramName = f"{getProjectNameFromPath(OriginalProjectPath)} ( Folder sizes )"; print(diagramName)
plot_sankey(*generate_sankey_from_tree_dfs(generate_directory_tree(OriginalProjectPath), display_mode=DisplayMode.SIZE,max_depth=5,skip_threshold=0.04),title=diagramName,height=500)
diagramName = f"{getProjectNameFromPath(OriginalProjectPath)} ( File count )"; print(diagramName)
plot_sankey(*generate_sankey_from_tree_dfs(generate_directory_tree(OriginalProjectPath), display_mode=DisplayMode.FILE_COUNT,max_depth=5,skip_threshold=0.1),title=diagramName,height=500)

2018 FreudMeOutProject ( Folder sizes )


2018 FreudMeOutProject ( File count )


In [19]:
plot_sankey(*generate_sankey_from_tree_dfs(generate_directory_tree(SecondProjectPath), display_mode=DisplayMode.SIZE,max_depth=None,skip_threshold=0.04),title=getProjectNameFromPath(SecondProjectPath) + " ( Folder sizes )",height=500)
plot_sankey(*generate_sankey_from_tree_dfs(generate_directory_tree(SecondProjectPath), display_mode=DisplayMode.FILE_COUNT,max_depth=5,skip_threshold=0.1),title=getProjectNameFromPath(SecondProjectPath) + " ( File count )",height=500)

In [ ]:
plot_sankey(*generate_sankey_from_tree_dfs(generate_directory_tree(MobileProjectPath), display_mode=DisplayMode.SIZE,max_depth=7,skip_threshold=0.04),title=getProjectNameFromPath(MobileProjectPath) + " ( Folder sizes )",height=600)
plot_sankey(*generate_sankey_from_tree_dfs(generate_directory_tree(MobileProjectPath), display_mode=DisplayMode.FILE_COUNT,max_depth=7,skip_threshold=0.07),title=getProjectNameFromPath(MobileProjectPath) + " ( File count )",height=600)

In [144]:
plot_sankey(*generate_sankey_from_tree_dfs(generate_directory_tree(VrProjectPath), display_mode=DisplayMode.SIZE,max_depth=9,skip_threshold=0.04),title=getProjectNameFromPath(VrProjectPath) + " ( Folder sizes )",height=600)
plot_sankey(*generate_sankey_from_tree_dfs(generate_directory_tree(VrProjectPath), display_mode=DisplayMode.FILE_COUNT,max_depth=9,skip_threshold=0.07),title=getProjectNameFromPath(VrProjectPath) + " ( File count )",height=600)